# LLaMA Factory를 이용한 Multi-GPU Full Fine-Tuning

**Qwen** 모델을 **LLaMA Factory** 프레임워크를 사용하여 Multi-GPU 환경에서 **Full Fine-Tuning**하는 방법을 학습합니다.

---

## 📋 학습 목표
- Single-node Multi-GPU 환경 구축 이해
- LLaMA Factory CLI를 사용한 Full Fine-Tuning 실행
- TensorBoard를 통한 학습 모니터링
- HuggingFace Hub에 모델 업로드

---

## 📚 참고 자료
- [LLaMA Factory - Distributed Training](https://llamafactory.readthedocs.io/en/latest/advanced/distributed.html)
- [Qwen Models - HuggingFace](https://huggingface.co/Qwen/models)
- [DeepSpeed Documentation](https://www.deepspeed.ai/)

---

## 1️⃣ Full Fine-Tuning vs QLoRA

### Full Fine-Tuning이란?
모델의 **모든 파라미터**를 학습하는 방식으로, 최상의 성능을 달성할 수 있지만 많은 GPU 메모리와 시간이 필요합니다.

### 비교표

| 항목 | QLoRA | Full Fine-Tuning |
|------|-------|------------------|
| **학습 파라미터** | 어댑터만 (~1%) | 전체 파라미터 (100%) |
| **GPU 메모리** | 낮음 (6GB~) | 높음 (40GB~) |
| **학습 시간** | 빠름 | 느림 |
| **성능** | 좋음 | 최상 |
| **GPU 요구사항** | 단일 GPU 가능 | Multi-GPU 권장 |

### 언제 Full Fine-Tuning을 사용하나요?
- ✅ 도메인이 크게 다른 경우 (의료, 법률 등)
- ✅ 최고의 성능이 필요한 경우
- ✅ 충분한 GPU 자원이 있는 경우
- ✅ 상업적 배포를 위한 모델

## 2️⃣ Multi-GPU 분산 학습 전략

LLaMA Factory는 세 가지 분산 학습 엔진을 지원합니다.

### 1. DDP (DistributedDataParallel)
- PyTorch 네이티브 분산 학습
- 각 GPU가 모델 전체 복사본 유지
- 간단하지만 메모리 효율 낮음

### 2. DeepSpeed
- Microsoft의 분산 학습 최적화 엔진
- **ZeRO** (Zero Redundancy Optimizer) 기술
- Offload, Sparse Attention 등 지원
- **✅ 이번 강의에서 사용**

### 3. FSDP (Fully Sharded Data Parallel)
- PyTorch의 고급 분산 학습 기법
- 모델 파라미터를 GPU 간 샤딩

---

### DeepSpeed ZeRO 단계

| 단계 | 샤딩 대상 | 메모리 절감 | 통신 오버헤드 |
|------|----------|------------|--------------|
| **ZeRO-0** | 없음 | 없음 | 최소 |
| **ZeRO-1** | Optimizer | 4배 | 낮음 |
| **ZeRO-2** | Optimizer + Gradient | 8배 | 중간 |
| **ZeRO-3** | Optimizer + Gradient + Model | 64배+ | 높음 |

> **권장**: 대부분의 경우 **ZeRO-2** 또는 **ZeRO-3**를 사용합니다.

## 3️⃣ Runpod 환경 설정

### 3.1 Runpod이란?
**Runpod**은 클라우드 GPU 렌탈 서비스로, 저렴한 가격에 고성능 GPU를 사용할 수 있습니다.

### 3.2 Pod 생성
1. [Runpod 웹사이트](https://www.runpod.io/) 접속
2. **Deploy** → **GPU Pod** 선택
3. GPU 선택 (예: 2x RTX 4090, 2x A100 등)
4. **Docker Image**에 커스텀 Dockerfile 사용

### 3.3 Dockerfile 확인

우리의 Dockerfile은 Multi-GPU 환경에 최적화되어 있습니다:

```dockerfile
FROM nvidia/cuda:12.1.1-devel-ubuntu22.04

# Multi-GPU 통신 최적화 환경 변수
ENV NCCL_P2P_DISABLE=0 \
    NCCL_IB_DISABLE=1 \
    UCX_TLS=tcp,self

# Python 3.11 및 필수 패키지 설치
RUN apt-get update && apt-get install -y \
    python3.11 python3.11-dev git wget vim

# PyTorch (CUDA 12.1)
RUN pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Flash-Attention (학습 속도 향상)
RUN MAX_JOBS=4 pip install flash-attn --no-build-isolation

# DeepSpeed (Multi-GPU 메모리 분산)
RUN pip install deepspeed==0.18.4

# LLaMA Factory
RUN pip install llamafactory[metrics,bitsandbytes,qwen]==0.9.1
```

### 주요 포인트
- **NCCL**: NVIDIA Collective Communications Library (GPU 간 통신)
- **DeepSpeed**: ZeRO 최적화를 위한 필수 라이브러리
- **Flash-Attention**: 학습 속도를 2~3배 향상

## 4️⃣ Qwen 모델 준비

### 4.1 Qwen 모델 소개
**Qwen**은 Alibaba Cloud가 개발한 고성능 오픈소스 LLM입니다.

| 모델 | 파라미터 | 권장 GPU |
|------|---------|---------|
| Qwen2.5-0.5B | 0.5B | 1x GPU (Full FT 가능) |
| Qwen2.5-1.5B | 1.5B | 1x GPU (Full FT 가능) |
| Qwen2.5-3B | 3B | 2x GPU (Full FT) |
| Qwen2.5-7B | 7B | 2~4x GPU (Full FT) |
| Qwen2.5-14B | 14B | 4~8x GPU (Full FT) |

### 4.2 모델 다운로드 (사전 작업)
모델은 미리 다운로드하여 로컬 디렉토리에 저장되어 있어야 합니다.

```python
from huggingface_hub import snapshot_download

# 예시: Qwen2.5-3B 모델 다운로드
model_name = "Qwen/Qwen2.5-3B-Instruct"
local_dir = "/workspace/models/Qwen2.5-3B-Instruct"

snapshot_download(
    repo_id=model_name,
    local_dir=local_dir,
    local_dir_use_symlinks=False
)
```

### 4.3 모델 경로 확인
학습 시 다음과 같이 경로를 지정합니다:
```bash
MODEL_PATH="/workspace/models/Qwen2.5-3B-Instruct"
```

## 5️⃣ 데이터셋 준비

### 5.1 데이터셋 확인
우리는 `illnesses.csv` 데이터셋을 사용합니다. 이 데이터셋은 의료 도메인 QA 데이터입니다.

In [ ]:
import pandas as pd

# 데이터셋 로드
df = pd.read_csv('data/illnesses.csv')

# 첫 3개 샘플 확인
print(f"총 샘플 수: {len(df)}")
print("\n=== 데이터셋 구조 ===")
print(df.head(3))

print("\n=== 컬럼 정보 ===")
print(df.columns.tolist())

### 5.2 LLaMA Factory 형식으로 변환

LLaMA Factory는 JSON 형식의 데이터셋을 요구합니다. 다음 포맷으로 변환합니다:

```json
[
  {
    "instruction": "질문 내용",
    "input": "",
    "output": "답변 내용"
  }
]
```

또는 대화형 포맷:
```json
[
  {
    "conversations": [
      {"role": "user", "content": "질문"},
      {"role": "assistant", "content": "답변"}
    ]
  }
]
```

In [ ]:
import json
from sklearn.model_selection import train_test_split

# 대화형 포맷으로 변환
def convert_to_conversations(row):
    return {
        "conversations": [
            {
                "role": "user",
                "content": row['user_input']
            },
            {
                "role": "assistant",
                "content": row['reference']
            }
        ]
    }

# 데이터 변환
dataset = [convert_to_conversations(row) for _, row in df.iterrows()]

# Train/Validation Split (90:10)
train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42)

print(f"학습 데이터: {len(train_data)}개")
print(f"검증 데이터: {len(val_data)}개")

# JSON 파일로 저장
with open('data/illnesses_train.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open('data/illnesses_val.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

# 샘플 확인
print("\n=== 변환된 데이터 샘플 ===")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

## 6️⃣ LLaMA Factory 설정 파일 작성

### 6.1 Dataset Info 등록
먼저 데이터셋을 LLaMA Factory에 등록해야 합니다. `dataset_info.json` 파일을 생성합니다.

In [ ]:
%%writefile dataset_info.json
{
  "illnesses_train": {
    "file_name": "data/illnesses_train.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant"
    }
  },
  "illnesses_val": {
    "file_name": "data/illnesses_val.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant"
    }
  }
}

### 6.2 학습 설정 YAML 파일
이제 Full Fine-Tuning을 위한 학습 설정 파일을 작성합니다.

In [ ]:
%%writefile train_full_ft.yaml
### ======================== 모델 설정 ========================
model_name_or_path: /workspace/models/Qwen2.5-3B-Instruct
trust_remote_code: true

### ======================== 학습 방법 ========================
stage: sft                    # Supervised Fine-Tuning
do_train: true
finetuning_type: full         # Full Fine-Tuning (모든 파라미터 학습)

### ======================== 데이터셋 ========================
dataset_dir: .
dataset: illnesses_train      # dataset_info.json에 등록한 이름
val_dataset: illnesses_val
template: qwen                # Qwen 모델용 템플릿
cutoff_len: 2048              # 최대 시퀀스 길이
max_samples: 1000             # (선택) 샘플 수 제한 (테스트용)
overwrite_cache: true
preprocessing_num_workers: 8

### ======================== 출력 설정 ========================
output_dir: ./outputs/qwen-3b-illnesses-full
overwrite_output_dir: true
logging_steps: 10
save_steps: 100
plot_loss: true

### ======================== 학습 하이퍼파라미터 ========================
per_device_train_batch_size: 2
per_device_eval_batch_size: 2
gradient_accumulation_steps: 8   # 유효 배치 사이즈 = 2 * 2 GPUs * 8 = 32
learning_rate: 2.0e-5            # Full FT는 낮은 LR 사용
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true                       # BFloat16 사용 (A100, H100, RTX 4090 등)

### ======================== 최적화 및 정규화 ========================
optim: adamw_torch
weight_decay: 0.01
max_grad_norm: 1.0

### ======================== 평가 설정 ========================
evaluation_strategy: steps
eval_steps: 100
load_best_model_at_end: true
metric_for_best_model: loss

### ======================== DeepSpeed 설정 ========================
deepspeed: configs/ds_config_zero2.json   # ZeRO-2 사용
ddp_timeout: 1800

### ======================== 기타 설정 ========================
report_to: tensorboard          # TensorBoard 로깅
save_total_limit: 2             # 최근 2개 체크포인트만 유지

### 6.3 DeepSpeed 설정 파일
프로젝트의 `configs/ds_config_zero2.json` 파일을 확인합니다.

In [ ]:
!cat configs/ds_config_zero2.json

## 7️⃣ Shell Script를 통한 학습 실행

### 7.1 학습 실행 스크립트 작성
LLaMA Factory CLI를 사용하여 Multi-GPU 학습을 실행하는 shell script를 작성합니다.

In [ ]:
%%writefile train_multi_gpu.sh
#!/bin/bash

# Multi-GPU Full Fine-Tuning 실행 스크립트
echo "=========================================="
echo "LLaMA Factory Multi-GPU Full Fine-Tuning"
echo "=========================================="

# GPU 정보 확인
echo -e "\n[GPU 정보]"
nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

# 환경 변수 설정
export CUDA_VISIBLE_DEVICES=0,1  # 사용할 GPU 지정 (0번, 1번 GPU)
export NCCL_P2P_DISABLE=0        # P2P 통신 활성화
export NCCL_IB_DISABLE=1         # InfiniBand 비활성화

# 학습 시작 시간 기록
START_TIME=$(date +%s)
echo -e "\n[학습 시작] $(date)"

# LLaMA Factory CLI 실행 (FORCE_TORCHRUN으로 Multi-GPU 모드 활성화)
FORCE_TORCHRUN=1 llamafactory-cli train train_full_ft.yaml

# 학습 종료 시간 기록
END_TIME=$(date +%s)
ELAPSED_TIME=$((END_TIME - START_TIME))
HOURS=$((ELAPSED_TIME / 3600))
MINUTES=$(((ELAPSED_TIME % 3600) / 60))
SECONDS=$((ELAPSED_TIME % 60))

echo -e "\n=========================================="
echo "[학습 완료] $(date)"
echo "소요 시간: ${HOURS}시간 ${MINUTES}분 ${SECONDS}초"
echo "=========================================="

### 7.2 스크립트 실행

In [ ]:
# 실행 권한 부여
!chmod +x train_multi_gpu.sh

# 스크립트 실행
!bash train_multi_gpu.sh

### 7.3 주요 파라미터 설명

| 환경 변수 | 설명 |
|----------|------|
| `CUDA_VISIBLE_DEVICES` | 사용할 GPU 지정 (예: 0,1은 2개 GPU) |
| `NCCL_P2P_DISABLE` | GPU 간 P2P 통신 설정 (0=활성화, 1=비활성화) |
| `FORCE_TORCHRUN` | LLaMA Factory에서 Multi-GPU 모드 강제 활성화 |

### 실행 원리
- **FORCE_TORCHRUN=1**: `llamafactory-cli`가 내부적으로 `torchrun`을 사용하여 멀티 프로세스 실행
- **torchrun**: PyTorch의 분산 학습 런처
- 각 GPU마다 별도의 프로세스가 생성되어 병렬 학습

## 8️⃣ TensorBoard로 학습 모니터링

### 8.1 TensorBoard 시작
학습 중 또는 학습 후 loss 그래프를 확인할 수 있습니다.

In [ ]:
# Jupyter Notebook에서 TensorBoard 실행
%load_ext tensorboard
%tensorboard --logdir ./outputs/qwen-3b-illnesses-full --port 6006

## 9️⃣ 학습 결과 확인

### 9.1 출력 디렉토리 구조

In [ ]:
!tree -L 2 ./outputs/qwen-3b-illnesses-full

학습 완료 후 다음과 같은 파일들이 생성됩니다:

```
outputs/qwen-3b-illnesses-full/
├── checkpoint-100/          # 첫 번째 체크포인트
│   ├── config.json
│   ├── model.safetensors   # 모델 가중치
│   ├── trainer_state.json
│   └── ...
├── checkpoint-200/          # 두 번째 체크포인트
├── runs/                    # TensorBoard 로그
├── trainer_log.jsonl        # 학습 로그
├── training_args.bin
└── all_results.json         # 최종 결과

### 9.2 학습 로그 확인

In [ ]:
import json

# 학습 로그 읽기
with open('./outputs/qwen-3b-illnesses-full/trainer_log.jsonl', 'r') as f:
    logs = [json.loads(line) for line in f]

# 최종 결과 확인
print("=== 학습 통계 ===")
print(f"총 스텝 수: {logs[-1].get('current_steps', 'N/A')}")
print(f"최종 Train Loss: {logs[-1].get('loss', 'N/A'):.4f}")
print(f"최종 Eval Loss: {logs[-1].get('eval_loss', 'N/A'):.4f}")
print(f"학습률: {logs[-1].get('learning_rate', 'N/A')}")

# Loss 변화 시각화
import matplotlib.pyplot as plt

train_losses = [log['loss'] for log in logs if 'loss' in log]
steps = list(range(len(train_losses)))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(steps, train_losses, label='Train Loss', color='blue')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.legend()

# Eval Loss
eval_logs = [log for log in logs if 'eval_loss' in log]
if eval_logs:
    eval_steps = [log['current_steps'] for log in eval_logs]
    eval_losses = [log['eval_loss'] for log in eval_logs]
    
    plt.subplot(1, 2, 2)
    plt.plot(eval_steps, eval_losses, label='Eval Loss', color='orange', marker='o')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Evaluation Loss')
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.show()

### 9.3 파인튜닝된 모델 테스트

학습된 모델을 로드하여 추론을 테스트합니다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 모델 및 토크나이저 로드
model_path = "./outputs/qwen-3b-illnesses-full"  # 최종 모델 또는 checkpoint-XXX
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

# 테스트 질문
test_question = "아이가 열이 나고 손과 발에 물집이 생겼어요. 어떻게 해야 하나요?"

# 프롬프트 구성 (Qwen 포맷)
messages = [
    {"role": "user", "content": test_question}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 추론
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

# 결과 출력
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== 질문 ===")
print(test_question)
print("\n=== 답변 ===")
print(response)

## 🔟 HuggingFace Hub에 모델 업로드

### 10.1 HuggingFace 로그인
먼저 HuggingFace 계정으로 로그인합니다.

In [ ]:
from huggingface_hub import login, HfApi

# HuggingFace 토큰으로 로그인
# 토큰은 https://huggingface.co/settings/tokens 에서 발급
HF_TOKEN = "your_huggingface_token_here"  # 여기에 본인의 토큰 입력

login(token=HF_TOKEN)
print("✅ HuggingFace 로그인 성공!")

### 10.2 모델 업로드

학습된 모델을 HuggingFace Hub에 업로드합니다.

In [ ]:
# 업로드할 모델 경로 및 리포지토리 설정
model_path = "./outputs/qwen-3b-illnesses-full"
repo_name = "your-username/qwen-3b-illnesses-full"  # 예: "johndoe/qwen-3b-medical-ko"

# 모델 및 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

# HuggingFace Hub에 업로드
print(f"📤 {repo_name} 리포지토리에 모델 업로드 중...")

model.push_to_hub(
    repo_name,
    private=False,  # True로 설정하면 비공개 모델
    commit_message="Upload fine-tuned Qwen model on medical dataset"
)

tokenizer.push_to_hub(
    repo_name,
    commit_message="Upload tokenizer"
)

print(f"✅ 업로드 완료!")
print(f"🔗 모델 URL: https://huggingface.co/{repo_name}")

### 10.3 CLI를 사용한 업로드 (대안)

CLI를 사용하여 전체 디렉토리를 업로드할 수도 있습니다.

In [ ]:
%%bash
# HuggingFace CLI로 로그인
# huggingface-cli login

# 전체 디렉토리 업로드
# huggingface-cli upload your-username/qwen-3b-illnesses-full ./outputs/qwen-3b-illnesses-full --repo-type model

### 10.4 Model Card 작성

업로드 후 HuggingFace Hub에서 모델 카드(README.md)를 작성하는 것이 좋습니다.

**Model Card에 포함할 내용:**

```markdown
# Qwen 3B - Korean Medical Illnesses Fine-tuned

## 모델 설명
이 모델은 Qwen2.5-3B-Instruct를 한국어 의료 도메인(소아청소년과) 데이터셋으로 Full Fine-Tuning한 모델입니다.

## 학습 정보
- **베이스 모델**: Qwen/Qwen2.5-3B-Instruct
- **파인튜닝 방법**: Full Fine-Tuning
- **데이터셋**: 한국어 소아청소년과 질병 QA (795 샘플)
- **학습 환경**: 2x GPU (DeepSpeed ZeRO-2)
- **학습 에폭**: 3 epochs
- **배치 사이즈**: 32 (effective)
- **학습률**: 2e-5

## 사용 방법
\`\`\`python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "your-username/qwen-3b-illnesses-full",
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("your-username/qwen-3b-illnesses-full")

messages = [
    {"role": "user", "content": "아이가 열이 나고 기침을 해요. 어떻게 해야 하나요?"}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
\`\`\`

## 제한 사항
- 의료 진단 목적으로 사용하지 마세요
- 실제 의료 상황에서는 반드시 전문의와 상담하세요
- 교육 및 연구 목적으로만 사용하세요

## 라이선스
Apache 2.0
```

## 1️⃣1️⃣ 전체 워크플로우 요약

### 단계별 프로세스

```mermaid
graph TD
    A[1. Runpod Pod 생성] --> B[2. Dockerfile로 환경 구축]
    B --> C[3. Qwen 모델 준비]
    C --> D[4. 데이터셋 변환 JSON]
    D --> E[5. dataset_info.json 작성]
    E --> F[6. train_full_ft.yaml 작성]
    F --> G[7. Shell Script 실행]
    G --> H[8. Multi-GPU 학습]
    H --> I[9. TensorBoard 모니터링]
    I --> J[10. 모델 테스트]
    J --> K[11. HuggingFace Hub 업로드]
```

### 명령어 체크리스트

✅ **환경 설정**
```bash
# Docker 빌드 및 실행은 Runpod에서 자동 처리
```

✅ **데이터 준비**
```python
# Notebook 셀 실행
- illnesses.csv 확인
- JSON 변환 및 Train/Val Split
- dataset_info.json 생성
```

✅ **학습 설정**
```yaml
# train_full_ft.yaml 작성
- 모델 경로 확인
- 하이퍼파라미터 조정
- DeepSpeed 설정 확인
```

✅ **학습 실행**
```bash
chmod +x train_multi_gpu.sh
bash train_multi_gpu.sh
```

✅ **모니터링**
```bash
tensorboard --logdir ./outputs/qwen-3b-illnesses-full --port 6006
```

✅ **업로드**
```python
model.push_to_hub("username/model-name")
tokenizer.push_to_hub("username/model-name")
```

## 1️⃣2️⃣ 트러블슈팅

### 자주 발생하는 문제와 해결 방법

#### 1. CUDA Out of Memory
```
RuntimeError: CUDA out of memory
```

**해결 방법:**
- `per_device_train_batch_size` 줄이기 (2 → 1)
- `gradient_accumulation_steps` 늘리기 (8 → 16)
- `cutoff_len` 줄이기 (2048 → 1024)
- DeepSpeed ZeRO-3 사용
- Flash-Attention 활성화 확인

#### 2. NCCL 통신 오류
```
NCCL error: unhandled system error
```

**해결 방법:**
```bash
export NCCL_P2P_DISABLE=1  # P2P 비활성화
export NCCL_IB_DISABLE=1   # InfiniBand 비활성화
export NCCL_DEBUG=INFO     # 디버그 로그 확인
```

#### 3. 학습 속도가 너무 느림
**해결 방법:**
- Flash-Attention 설치 확인
- `preprocessing_num_workers` 증가 (8 → 16)
- `dataloader_num_workers` 추가
- 더 빠른 GPU 사용 (A100, H100)

#### 4. Loss가 NaN으로 변함
**해결 방법:**
- Learning rate 낮추기 (2e-5 → 5e-6)
- `max_grad_norm` 낮추기 (1.0 → 0.5)
- `weight_decay` 조정
- 데이터셋 확인 (이상치 제거)

#### 5. 모델이 수렴하지 않음
**해결 방법:**
- Learning rate 높이기
- Warmup steps 늘리기
- 더 많은 epoch 학습
- 데이터셋 품질 확인

## 1️⃣3️⃣ 하이퍼파라미터 튜닝 가이드

### Full Fine-Tuning 권장 하이퍼파라미터

| 파라미터 | 작은 모델 (3B) | 중간 모델 (7B) | 큰 모델 (14B+) |
|---------|--------------|--------------|--------------|
| **Learning Rate** | 2e-5 ~ 5e-5 | 1e-5 ~ 3e-5 | 5e-6 ~ 2e-5 |
| **Batch Size** | 16 ~ 32 | 8 ~ 16 | 4 ~ 8 |
| **Epochs** | 3 ~ 5 | 2 ~ 4 | 1 ~ 3 |
| **Warmup Ratio** | 0.1 | 0.05 ~ 0.1 | 0.03 ~ 0.05 |
| **Weight Decay** | 0.01 ~ 0.1 | 0.01 ~ 0.1 | 0.01 |
| **Max Grad Norm** | 1.0 | 1.0 | 0.5 ~ 1.0 |

### 데이터셋 크기별 가이드

| 데이터셋 크기 | Epoch | Learning Rate | 비고 |
|-------------|-------|--------------|------|
| **< 1K** | 5 ~ 10 | 높음 (5e-5) | 과적합 주의 |
| **1K ~ 10K** | 3 ~ 5 | 중간 (2e-5) | 권장 |
| **10K ~ 100K** | 2 ~ 3 | 낮음 (1e-5) | 안정적 |
| **100K+** | 1 ~ 2 | 매우 낮음 (5e-6) | 충분한 학습 |

### Scheduler 선택

| Scheduler | 특징 | 권장 상황 |
|-----------|------|----------|
| **cosine** | 부드러운 감소 | 대부분의 경우 (권장) |
| **linear** | 선형 감소 | 짧은 학습 |
| **constant** | 고정 LR | 실험용 |
| **polynomial** | 다항식 감소 | 긴 학습 |

## 1️⃣4️⃣ 참고 자료 및 추가 학습

### 공식 문서
- 📚 [LLaMA Factory Documentation](https://llamafactory.readthedocs.io/)
- 📚 [Qwen Models - HuggingFace](https://huggingface.co/Qwen/models)
- 📚 [DeepSpeed ZeRO Tutorial](https://www.deepspeed.ai/tutorials/zero/)
- 📚 [Transformers Distributed Training](https://huggingface.co/docs/transformers/main/en/perf_train_gpu_many)

### 추천 읽을거리
- 📖 [LoRA vs Full Fine-Tuning 비교](https://arxiv.org/abs/2106.09685)
- 📖 [DeepSpeed ZeRO 논문](https://arxiv.org/abs/1910.02054)
- 📖 [Qwen2 Technical Report](https://arxiv.org/abs/2407.10671)

### 관련 도구
- 🛠️ [vLLM - 추론 최적화](https://github.com/vllm-project/vllm)
- 🛠️ [llama.cpp - GGUF 변환](https://github.com/ggerganov/llama.cpp)
- 🛠️ [Axolotl - 대안 파인튜닝 프레임워크](https://github.com/OpenAccess-AI-Collective/axolotl)

### 다음 단계

1️⃣ **모델 최적화**
- GGUF 형식으로 변환 (양자화)
- vLLM으로 추론 서버 구축
- OpenAI API 호환 엔드포인트 생성

2️⃣ **고급 파인튜닝**
- DPO (Direct Preference Optimization)
- RLHF (Reinforcement Learning from Human Feedback)
- Continual Pre-training

3️⃣ **프로덕션 배포**
- Docker 컨테이너화
- Kubernetes 배포
- 모니터링 및 로깅 시스템 구축

## 🎓 요약 및 결론

### 핵심 내용 정리

✅ **Full Fine-Tuning의 이해**
- 모든 파라미터를 학습하여 최상의 성능 달성
- QLoRA보다 많은 GPU 메모리 필요
- 도메인 특화 모델 개발에 최적

✅ **Multi-GPU 분산 학습**
- DeepSpeed ZeRO로 메모리 효율성 극대화
- NCCL을 통한 GPU 간 통신 최적화
- `FORCE_TORCHRUN`으로 간편한 멀티 프로세스 실행

✅ **LLaMA Factory CLI 활용**
- Shell Script를 통한 자동화된 학습 파이프라인
- YAML 설정 파일로 체계적인 하이퍼파라미터 관리
- TensorBoard 통합으로 실시간 모니터링

✅ **실전 프로덕션 워크플로우**
- Runpod + Docker로 재현 가능한 환경 구축
- 데이터셋 준비부터 HuggingFace 배포까지 전체 파이프라인
- 의료 도메인 등 특화 분야 모델 개발 역량

### 배운 것들
1. ✨ Full Fine-Tuning과 QLoRA의 차이점 및 선택 기준
2. 🚀 DeepSpeed ZeRO를 사용한 Multi-GPU 학습 최적화
3. 📊 TensorBoard를 통한 학습 모니터링 및 분석
4. 🎯 Qwen 모델 파인튜닝 및 의료 도메인 적용
5. 🌐 HuggingFace Hub를 통한 모델 공유 및 배포

---

## 🎉 수고하셨습니다!

본 강의를 통해 sLLM을 Multi-GPU 환경에서 Full Fine-Tuning하는 전체 프로세스를 학습하셨습니다.

이제 여러분만의 데이터셋으로 도메인 특화 모델을 개발하고, 실제 프로덕션 환경에 배포할 수 있습니다! 🚀

**다음 강의에서 만나요!**